# Zutaten-Normalisierung & Gruppierung mit NLP

## Zielsetzung
Erstellung eines Ähnlichkeits-Caches für Zutaten aus TheMealDB, um:
- **Plural/Singular** zusammenzufassen (z.B. "Tomato" ↔ "Tomatoes")
- **Varianten** zu erkennen (z.B. "Cherry Tomatoes" ↔ "Baby Plum Tomatoes")
- **Modifikatoren** zu behandeln (z.B. "Fresh Basil" → "Basil")

---

## Workflow-Übersicht

```
1. Zutaten-Pool laden
    ↓
2. Grammatik-Analyse (spaCy)
    ↓
3. Token-Cache erstellen
    ↓
4. Ähnlichkeits-Vergleiche
    ↓
5. JSON-Export
```

---

## 1. Grammatik-Analyse mit spaCy

### Kernproblem: "Tomato Sauce" vs. "Tomatoes"
- **HEAD**: Grammatikalisches Zentrum (→ `sauce` / `tomato`)
- **MODIFIER**: Erweiterte Information (→ `tomato` / ∅)

### Manuelle Korrekturen
```python
NOISE_WORDS = {"leaves", "leaf", "seed", "nuts", "yolks"}
MANUAL_CORRECTIONS = {"leaves": "leaf", "nuts": "nut", "yolks": "yolk"}
```

### Funktion: `analyze_grammar(text)`
- Lemmatisierung (z.B. "tomatoes" → "tomato")
- Trennung von HEAD und MODIFIERS
- Ignoriert: Adjektive (`amod`), Präpositionen (`prep`)

**Beispiel:**
```
"Fresh Cherry Tomatoes"
→ HEAD: "tomato"
→ MODIFIER: "cherry"
(Ignoriert: "fresh" als Adjektiv)
```

---

## 2. Token-Vektorisierung

### Vektorkombination (4 Methoden)
Jede Zutat wird in 2 Vektoren zerlegt (HEAD + MODIFIER) und kombiniert:

| Methode | Formel | Vorteil |
|---------|--------|---------|
| `weighted` | 70% HEAD + 30% MOD | Betont grammatikalische Hierarchie |
| `concat` | [HEAD \|\| MOD] | Keine Informationsverlust (600D) |
| `hadamard` | HEAD ⊙ MOD | Betont gemeinsame Features |
| `max` | max(HEAD, MOD) | Robusteste Features beider Vektoren |

**Aktuell genutzt:** `METHOD = "max"`

---

## 3. IngredientTokenCache

### Initialisierung
```python
cache = IngredientTokenCache(ALL_INGREDIENTS, method="max")
```

**Workflow:**
1. Für jede Zutat: `analyze_grammar()` → HEAD + MOD
2. Konvertierung zu spaCy-Tokens (`en_core_web_md` für Vektoren)
3. Kombination zu `combine_tokens[ingredient]`

**Vorteil:** Einmalige Berechnung statt 877² Vergleiche!

---

## 4. Ähnlichkeits-Vergleich

### Funktion: `check_similarity_combined()`
- **Cosinus-Ähnlichkeit** zwischen kombinierten Vektoren
- Threshold: `0.85` (anpassbar)
- Output: `[(ingredient, score), ...]` (sortiert nach Score)

**Beispiel:**
```python
cache.check_similarity("Cucumber", threshold=0.85)
# → {"Cucumber": [("Cucumbers", 0.98), ("Zucchini", 0.87), ...]}
```

---

## 5. JSON-Export

### Dateiformat
```json
{
  "Chicken": [
     ["Chicken Breast", 0.95],
     ["Chicken Legs", 0.92],
     ["Chicken Thighs", 0.91]
  ],
  "Tomato": [
     ["Tomatoes", 0.99],
     ["Cherry Tomatoes", 0.89],
     ["Baby Plum Tomatoes", 0.87]
  ]
}
```

### Dateiname
```
ingredient_similarity_cache_DD-MM-YYYY-HH-MM_{METHOD}.json
```

**Generierung für alle Methoden:**
```python
for method in ["weighted", "concat", "hadamard", "max"]:
     write_JSON_similar_ingredients_fast(method)
```

---

## Validierung

### Funktion: `validate_similarity_cache()`
Prüft:
- ✅ Alle 877 Zutaten haben einen Key
- ✅ Keine leeren Listen
- ⚠️ Identifiziert problematische Fälle (z.B. zu hohe Thresholds)

---

## Probleme & Lösungen

| Problem | Ursache | Lösung |
|---------|---------|--------|
| Plural/Singular getrennt | spaCy erkennt nicht immer Lemma | Manuelle `MANUAL_CORRECTIONS` |
| "Leaves"→Lemma "Leave ≠ "Leaf" | Noise-Word-Handling | `NOISE_WORDS` Set |
| "Basil Leave" → "Basil" | Modifier dominiert | `NOISE_WORDS` Set |
| "Walnut Oil" → "Oil" | HEAD dominiert | Modifier-Vektor mit 30% gewichtet |
| JSON-Serialisierung scheitert | `numpy.float32` | Konvertierung zu Python-`float()` |

---

## Performance

- **Ohne Cache:** ~385.000 NLP-Vergleiche (877²)
- **Mit Cache:** ~877 Vergleiche (1× Initialisierung + 1× Lookup)
- **Speedup:** ~440×

In [19]:
import os
import sys

# Pfad zum übergeordneten Verzeichnis hinzufügen (damit themealdb_client gefunden wird)
sys.path.insert(0, os.path.abspath('..'))

from themealdb_client import TheMealDBClient
import json
import traceback

# Test für get_all_ingredients() Funktion

# Client initialisieren
client = TheMealDBClient()
def get_all_ingredients():
    try:
        ingredients = client.get_all_ingredients()
        ingredients = [ingredient['strIngredient'] for ingredient in ingredients if 'strIngredient' in ingredient]
        print(f"Anzahl Zutaten gefunden: {len(ingredients)}")
        print("Beispiel-Zutaten:")
        for ing in ingredients[:10]:  # Zeige die ersten 10 Zutaten
            print(f"- {ing}")
        return ingredients
    except Exception as e:
        print("Fehler beim Abrufen der Zutaten:")
        traceback.print_exc()
        return []
ALL_INGREDIENTS = get_all_ingredients()


Anzahl Zutaten gefunden: 877
Beispiel-Zutaten:
- Chicken
- Salmon
- Beef
- Pork
- Avocado
- Apple Cider Vinegar
- Asparagus
- Aubergine
- Baby Plum Tomatoes
- Bacon


In [35]:
# Globale Liste an "harmlosen" Köpfen (Container/Zustände),
# die den eigentlichen Kern einer Zutat nicht verändern.
# Beispiel: "Basil Leaves" soll zu "Basil" passen.
SAFE_HEADS = ["leaf", "leaves", "slice", "slices", "piece", "pieces", "wedge", "clove", "cloves", "seed", "seeds"]

class IngredientSimilarityChecker:
    """
    Bewertet Zutaten-Paare semantisch (SentenceTransformer) + strukturell (spaCy-Head).
    Optimiert für Wiederverwendung: Modelle + Embeddings werden nur einmal geladen.
    """

    def __init__(self, ingredient_list):
        """
        Initialisiert die Klasse, lädt Modelle und erstellt Embeddings.

        Schritte:
        1) spaCy-Modell laden (für Grammatik/Head-Erkennung)
        2) SentenceTransformer laden (für semantische Ähnlichkeit)
        3) Levenshtein-Ratio vorbereiten (für String-Ähnlichkeit)
        4) Embeddings für die Zutatenliste vorab berechnen
        5) Head-Nouns für die Zutatenliste vorab berechnen
        """
        import spacy
        from sentence_transformers import SentenceTransformer, util
        from Levenshtein import ratio

        # Zutatenliste speichern
        self.ingredient_list = ingredient_list

        # 1) spaCy-Modelle laden (md bevorzugt für Vektoren)
        try:
            self.nlp = spacy.load("en_core_web_md")
        except:
            self.nlp = spacy.load("en_core_web_sm")

        # 2) Transformer-Modell für semantische Ähnlichkeit
        self.model = SentenceTransformer('all-MiniLM-L6-v2')

        # 3) Utilities zwischenspeichern
        self.util = util
        self.ratio = ratio

        # Normalisierte Zutatenliste (einheitliche Keys)
        normalized_ingredients = [self._normalize_text(ing) for ing in ingredient_list]
        normalized_unique = list(dict.fromkeys(normalized_ingredients))

        # 4) Embeddings für alle Zutaten vorberechnen (einmalig)
        self.embeddings = {
            ing_norm: self.model.encode(ing_norm, convert_to_tensor=True)
            for ing_norm in normalized_unique
        }

        # 5) Head-Nouns für alle Zutaten vorberechnen (einmalig)
        self.head_nouns = {
            ing_norm: self._compute_head_noun(ing_norm)
            for ing_norm in normalized_unique
        }

    def _normalize_text(self, text):
        """
        Normalisiert Zutaten-Strings konsistent für Cache-Keys.
        - lower()
        - Mehrfache Spaces entfernen
        - Leading/Trailing Spaces trimmen
        """
        return " ".join(text.lower().split())

    def _compute_head_noun(self, text):
        """
        Berechnet das grammatikalische Kopf-Nomen eines Ausdrucks.
        Beispiel: "Chicken Liver" -> "liver" (im Englischen steht der Head meist rechts).
        """
        doc = self.nlp(text.lower())

        # Alle Nomen/Proper-Nomen sammeln
        nouns = [t.lemma_ for t in doc if t.pos_ in ["NOUN", "PROPN"]]

        # Falls vorhanden, nimm das letzte Nomen als Head
        if nouns:
            return nouns[-1]

        # Fallback: letztes Token lemmatisiert
        return doc[-1].lemma_

    def get_head_noun(self, text):
        """
        Liefert das Head-Nomen aus dem Cache, falls vorhanden.
        Fallback: berechnet es on-the-fly für unbekannte Zutaten.
        """
        key = self._normalize_text(text)
        cached = self.head_nouns.get(key)
        if cached is not None:
            return cached
        return self._compute_head_noun(key)

    def _get_embedding(self, ingredient):
        """
        Liefert ein vorhandenes Embedding aus dem Cache.
        Wenn die Zutat nicht im Cache ist, wird es on-the-fly erzeugt.
        """
        key = self._normalize_text(ingredient)
        emb = self.embeddings.get(key)
        if emb is None:
            emb = self.model.encode(key, convert_to_tensor=True)
        return emb

    def similarity(self, ing_a, ing_b):
        """
        Hauptlogik für den Vergleich zweier Zutaten.

        Schritte:
        1) Strings normalisieren
        2) Grammatik-Check: Head-Nomen vergleichen
        3) Substring-Regel behandeln ("Tomato" in "Tinned Tomatoes")
        4) Semantik-Score via Transformer berechnen
        5) String-Score via Levenshtein berechnen
        6) Dynamischen Threshold bestimmen und entscheiden
        """
        # 1) Normalisierung für String-Vergleiche
        a_norm = self._normalize_text(ing_a)
        b_norm = self._normalize_text(ing_b)

        # 2) Head-Nomen bestimmen
        head_a = self.get_head_noun(a_norm)
        head_b = self.get_head_noun(b_norm)

        # 3) Substring-Prüfung (A ist Teil von B oder umgekehrt)
        is_substring = a_norm in b_norm or b_norm in a_norm
        if is_substring:
            # Wenn Heads unterschiedlich sind, ist es oft KEIN Match
            if head_a != head_b:
                # Ausnahme: SAFE_HEADS erlauben Varianten wie "Basil Leaves"
                if head_a in SAFE_HEADS or head_b in SAFE_HEADS:
                    return True, 1.0, "Safe Head Match"
                return False, 0.0, f"Head Mismatch ({head_a} != {head_b})"

            # Heads gleich -> sicherer Match
            return True, 1.0, "Substring & Head Match"

        # 4) Semantische Ähnlichkeit (Transformer)
        emb1 = self._get_embedding(a_norm)
        emb2 = self._get_embedding(b_norm)
        sem_score = float(self.util.cos_sim(emb1, emb2).item())

        # 5) String-Ähnlichkeit (Levenshtein)
        str_score = self.ratio(a_norm, b_norm)

        # 6) Dynamischer Threshold
        # Hoher String-Score -> Semantik darf etwas niedriger sein
        if str_score > 0.8:
            required_score = 0.65
        elif str_score > 0.6:
            required_score = 0.75
        else:
            required_score = 0.80

        # Entscheidung
        if sem_score > required_score:
            return True, sem_score, f"Semantic Match with Dynamic Threshold {required_score:.2f}"
        return False, sem_score, "Low Similarity"


In [37]:

# Beispielnutzung:
ingredients = [
    "Aubergine", "Eggplant", "Broccoli", "Cabbage",
    "Tomato", "Tinned Tomatoes", "Chicken", "Chicken Liver",
    "Basil", "Basil Leaves"
]
checker = IngredientSimilarityChecker(ingredients)

pairs = [
    ("Tomato", "Tinned Tomatoes"),
    ("Chicken", "Chicken Liver"),
    ("Basil", "Basil Leaves"),
    ("Basil", "fresh Basil Leaves"),
    ("Tomato", "tomato sauce")
]

print(f"{'A':<15} | {'B':<15} | {'Resultat':<10} | {'Grund'}")
print("-" * 60)
for a, b in pairs:
    match, score, reason = checker.similarity(a, b)
    icon = "✅" if match else "❌"
    print(f"{a:<15} | {b:<15} | {icon} {match:<6} | {reason} (Score: {score:.2f})")


A               | B               | Resultat   | Grund
------------------------------------------------------------
Tomato          | Tinned Tomatoes | ✅ 1      | Substring & Head Match (Score: 1.00)
Chicken         | Chicken Liver   | ❌ 0      | Head Mismatch (chicken != liver) (Score: 0.00)
Basil           | Basil Leaves    | ✅ 1      | Substring & Head Match (Score: 1.00)
Basil           | fresh Basil Leaves | ✅ 1      | Substring & Head Match (Score: 1.00)
Tomato          | tomato sauce    | ❌ 0      | Head Mismatch (tomato != sauce) (Score: 0.00)


In [33]:
def build_ingredient_similarity_json():
    """
    Erstellt ein Ähnlichkeits-Dict für alle Zutaten aus ALL_INGREDIENTS.
    Vergleicht jede Zutat mit allen anderen Zutaten.
    Speichert nur Matches (match==True) mit Gründen in eine JSON-Datei.
    
    Format:
    {
        "Zutat A": [
            ["Zutat B", "Substring & Head Match (Score: 1.00)"],
            ["Zutat C", "Semantic Match (Score: 0.78)"]
        ]
    }
    
    Dateiname: ingredient_similarity_json_DD-MM-YYYY-HH-MM.json
    """
    from datetime import datetime
    import json
    
    print(f"Initialisiere IngredientSimilarityChecker für {len(ALL_INGREDIENTS)} Zutaten...")
    checker_full = IngredientSimilarityChecker(ALL_INGREDIENTS)
    
    similarity_dict = {}
    
    print(f"\nVergleiche alle Zutaten-Paare...")
    total_pairs = len(ALL_INGREDIENTS)
    
    for idx, ingredient_a in enumerate(ALL_INGREDIENTS):
        if idx % 50 == 0:
            print(f"Fortschritt: {idx}/{total_pairs}")
        
        # Vergleiche mit allen anderen Zutaten
        for ingredient_b in ALL_INGREDIENTS:
            # Ähnlichkeit berechnen
            match, score, reason = checker_full.similarity(ingredient_a, ingredient_b)
            
            # Nur speichern, wenn Match == True
            if match:
                if ingredient_a not in similarity_dict:
                    similarity_dict[ingredient_a] = []
                similarity_dict[ingredient_a].append([ingredient_b, f"{reason} (Score: {score:.2f})"])
    
    # Dateiname mit Zeitstempel erstellen
    timestamp = datetime.now().strftime("%d-%m-%Y-%H-%M")
    output_file = f"ingredient_similarity_semantic_{timestamp}.json"
    
    # JSON speichern
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(similarity_dict, f, indent=2, ensure_ascii=False)
    
    print(f"\n✅ Fertig! Datei gespeichert: {output_file}")
    print(f"Zutaten mit Matches: {len(similarity_dict)}")
    print(f"Gesamte Matches: {sum(len(matches) for matches in similarity_dict.values())}")
    
    return similarity_dict

# Ausführen
similarity_results = build_ingredient_similarity_json()


Initialisiere IngredientSimilarityChecker für 877 Zutaten...

Vergleiche alle Zutaten-Paare...
Fortschritt: 0/877
Fortschritt: 50/877
Fortschritt: 100/877
Fortschritt: 150/877
Fortschritt: 200/877
Fortschritt: 250/877
Fortschritt: 300/877
Fortschritt: 350/877
Fortschritt: 400/877
Fortschritt: 450/877
Fortschritt: 500/877
Fortschritt: 550/877
Fortschritt: 600/877
Fortschritt: 650/877
Fortschritt: 700/877
Fortschritt: 750/877
Fortschritt: 800/877
Fortschritt: 850/877

✅ Fertig! Datei gespeichert: ingredient_similarity_semantic_31-01-2026-16-44.json
Zutaten mit Matches: 877
Gesamte Matches: 1981


In [34]:
def validate_similarity_cache(json_file, all_ingredients_list):
    """
    Validiert die erstellte JSON-Datei:
    - Prüft, ob jede Zutat einen Key hat
    - Prüft, ob die Listen nicht leer sind
    """
    try:
        with open(json_file, "r", encoding="utf-8") as f:
            cache_data = json.load(f)
    except Exception as e:
        print(f"Fehler beim Lesen der Datei: {e}")
        return False
    
    print(f"Validiere {json_file}...")
    print(f"Zutaten in ALL_INGREDIENTS: {len(all_ingredients_list)}")
    print(f"Keys in JSON: {len(cache_data)}")
    
    missing_keys = []
    empty_lists = []
    
    # Prüfe, ob alle Zutaten einen Key haben
    for ingredient in all_ingredients_list:
        if ingredient not in cache_data:
            missing_keys.append(ingredient)
        elif not cache_data[ingredient]:  # Liste ist leer
            empty_lists.append(ingredient)
    
    # Ausgabe der Ergebnisse
    print("\n--- VALIDIERUNGSERGEBNISSE ---")
    
    if missing_keys:
        print(f"\n❌ FEHLER: {len(missing_keys)} Zutaten ohne Key:")
        for ing in missing_keys[:10]:  # Zeige nur die ersten 10
            print(f"  - {ing}")
        if len(missing_keys) > 10:
            print(f"  ... und {len(missing_keys) - 10} mehr")
    else:
        print("✅ Alle Zutaten haben einen Key")
    
    if empty_lists:
        print(f"\n⚠️ WARNUNG: {len(empty_lists)} Zutaten haben leere Listen:")
        for ing in empty_lists[:10]:  # Zeige nur die ersten 10
            print(f"  - {ing}")
        if len(empty_lists) > 10:
            print(f"  ... und {len(empty_lists) - 10} mehr")
    else:
        print("✅ Keine leeren Listen gefunden")
    
    # Zusammenfassung
    success = len(missing_keys) == 0 and len(empty_lists) == 0
    print(f"\n{'✅ VALIDIERUNG ERFOLGREICH' if success else '❌ VALIDIERUNG FEHLGESCHLAGEN'}")
    
    return success

# # Beispielnutzung (nach Ausführung von write_JSON_similar_ingredients_fast()):
validate_similarity_cache("used_similarity_cache/ingredient_similarity_semantic_31-01-2026-16-44.json", ALL_INGREDIENTS)


Validiere used_similarity_cache/ingredient_similarity_semantic_31-01-2026-16-44.json...
Zutaten in ALL_INGREDIENTS: 877
Keys in JSON: 877

--- VALIDIERUNGSERGEBNISSE ---
✅ Alle Zutaten haben einen Key
✅ Keine leeren Listen gefunden

✅ VALIDIERUNG ERFOLGREICH


True